In [295]:
import pandas as pd
import numpy as np
import torch
import matplotlib.pyplot as plt

# **ASTR4004 - Problem Set 4**

Design a neural network for regression: with at least 5 neurons; involving either relu ($\mathrm{max}(0,\,x)$) or sigmoid ($1\,/\,(\exp(-x) + 1)$) activation functions; having a loss that, within 10 steps, monotonically decreases when the learning rate is 0.1 or increases when the learning rate is $>0.1$.

Read in data (6000 K/M-dwarf spectra and [M/H]).

In [296]:
def parse_flux(cell_string):
    # converts strings of spectra arrays to actual arrays
    cleaned = cell_string.strip("[]'\" \t\n")
    return np.fromstring(cleaned, sep=",")

full_data = pd.read_csv("./lowmass_spectra.csv", converters={"flux": parse_flux})
# remove nan metallicities
full_data = full_data[full_data["mh_gspspec"].notna()]

x_train = full_data["flux"] # input spectra for training
x_train_matrix = np.stack(x_train.values) 
x_train = torch.tensor(x_train_matrix, dtype=torch.float32)
x_train = torch.nan_to_num(x_train, nan=1.0, posinf=1.0, neginf=1.0)

y_train = full_data["mh_gspspec"] # metallicities for training
y_train = torch.tensor(y_train.values, dtype=torch.float32)
y_train = y_train.view(-1, 1)

Define network structure.

In [297]:
n_neurons = int(2**9)

class MHRegression(torch.nn.Module):

    def __init__(self):
        super(MHRegression, self).__init__() # add features of parent class to mh_regression

        # normalisation as suggested in lectures
        self.batch_norm = torch.nn.BatchNorm1d(num_features=2401)

        # hidden layer
        self.hidden_layer = torch.nn.Linear(in_features=2401, out_features=n_neurons)
        self.relu = torch.nn.ReLU() # ReLU activation function

        # output layer
        self.output_layer = torch.nn.Linear(in_features=n_neurons, out_features=1) # output scalar

    def forward(self, x):

        # evaluate z(1)
        z1 = self.hidden_layer(x)

        # evaluate y(1)
        y1 = self.relu(z1)

        # evaluate y(2) = z(2)
        y2 = self.output_layer(y1)

        return y2

model = MHRegression()

Define learning rate and loss function.

In [298]:
loss_func = torch.nn.MSELoss() # mean square error loss

eta = 1 # learning rate
optimise = torch.optim.SGD(model.parameters(), lr=eta)

Run training epochs.

In [299]:
epochs = 10
print(f"eta = {eta:.2f}")
loss_arr = np.zeros(epochs)
for epoch in range(epochs+1):
    optimise.zero_grad() # remove computed gradients from previous epoch
    # hidden layer
    h1 = model.hidden_layer(x_train)
    r1 = model.relu(h1)
    # output layer
    yhat = model.output_layer(r1)
    # compute loss (MSE)
    loss = loss_func(yhat, y_train)
    loss.backward() # calculate all neuron gradients
    
    with torch.no_grad():
        # hidden layer
        w1_val   = model.hidden_layer.weight.mean().item()
        b1_val   = model.hidden_layer.bias.mean().item()
        h1_val   = h1.mean().item()
        r1_val   = r1.mean().item()
        grad_w1  = model.hidden_layer.weight.grad.mean().item()
        grad_b1  = model.hidden_layer.bias.grad.mean().item()
        # output layer
        w2_val   = model.output_layer.weight.mean().item()
        b2_val   = model.output_layer.bias.mean().item()
        yhat_val = yhat.mean().item()
        grad_w2  = model.output_layer.weight.grad.mean().item()
        grad_b2  = model.output_layer.bias.grad.mean().item()
        # loss
        current_loss = loss.item()
        
    optimise.step() # update weights considering loss function and learning rate

    if epoch > 0:
        print(f"Epoch: {epoch:.0f}\tLoss: {loss.item():.4f}")
        print(
            f"{epoch} & {w1_val:.4f} & {b1_val:.4f} & {h1_val:.4f} & {r1_val:.4f} & {grad_w1:.4f} & {grad_b1:.4f} & "
            f"{w2_val:.4f} & {b2_val:.4f} & {yhat_val:.4f} & {grad_w2:.4f} & {grad_b2:.4f} & "
            f"{current_loss:.2e} \\\\"
        )
        
        loss_arr[epoch-1] = loss.item()

eta = 1.00
Epoch: 1	Loss: 45922904.0000
1 & -0.0008 & -0.0008 & -1.8523 & 18.6326 & 2197.5864 & 2352.6587 & -0.3865 & -1.6665 & -6774.9751 & -252613.2812 & -13548.9746 & 4.59e+07 \\
Epoch: 2	Loss: 183577712.0000
2 & -2197.5872 & -2352.6597 & -4986166.5000 & 0.0452 & 2.7522 & 2.9483 & 252612.9062 & 13547.3076 & 13548.5938 & 1224.1267 & 27098.1699 & 1.84e+08 \\
Epoch: 3	Loss: 183612608.0000
3 & -2200.3394 & -2355.6079 & -4992411.0000 & 0.0000 & 0.0000 & 0.0000 & 251388.7812 & -13550.8623 & -13550.8594 & 0.0000 & -27100.7461 & 1.84e+08 \\
Epoch: 4	Loss: 183612592.0000
4 & -2200.3394 & -2355.6079 & -4992411.0000 & 0.0000 & 0.0000 & 0.0000 & 251388.7812 & 13549.8838 & 13549.8896 & 0.0000 & 27100.7461 & 1.84e+08 \\
Epoch: 5	Loss: 183612608.0000
5 & -2200.3394 & -2355.6079 & -4992411.0000 & 0.0000 & 0.0000 & 0.0000 & 251388.7812 & -13550.8623 & -13550.8594 & 0.0000 & -27100.7461 & 1.84e+08 \\
Epoch: 6	Loss: 183612592.0000
6 & -2200.3394 & -2355.6079 & -4992411.0000 & 0.0000 & 0.0000 & 0.0000 